In [5]:
!pip install dash plotly scikit-learn pandas dash-bootstrap-components
# IMPORTS
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import plotly.express as px
import plotly.graph_objects as go
import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import dash_bootstrap_components as dbc
import warnings

warnings.filterwarnings('ignore')

# DATA LOADING & PREPARATION
csv_files = {
    '/content/AshokVihar_Hourly.csv': 'Ashok Vihar',
    '/content/DCStadium_Hourly.csv': 'DC Stadium',
    '/content/DwarkaSec8_Hourly.csv': 'Dwarka Sec 8',
    '/content/Najafgarh_Hourly.csv': 'Najafgarh',
    '/content/NehruNagar_Hourly.csv': 'Nehru Nagar',
    '/content/Okhla_Hourly.csv': 'Okhla'
}

delhi_coords = {
    'Ashok Vihar': {'lat': 28.695, 'lon': 77.181},
    'DC Stadium': {'lat': 28.611, 'lon': 77.237},
    'Dwarka Sec 8': {'lat': 28.571, 'lon': 77.071},
    'Najafgarh': {'lat': 28.609, 'lon': 76.979},
    'Nehru Nagar': {'lat': 28.567, 'lon': 77.251},
    'Okhla': {'lat': 28.524, 'lon': 77.284}
}

dataframes = []
for file_path, location in csv_files.items():
    if os.path.exists(file_path):
        df = pd.read_csv(file_path)
        if all(col in df.columns for col in ['year', 'month', 'day', 'hour']):
            df['Datetime'] = pd.to_datetime(df[['year', 'month', 'day', 'hour']])
            df = df.drop(columns=['year', 'month', 'day', 'hour'])
        df['Location'] = location
        dataframes.append(df)
    else:
        print(f"Warning: {file_path} not found. Skipping.")

combined_df = pd.concat(dataframes, ignore_index=True)
combined_df = combined_df.sort_values(by=['Location', 'Datetime']).reset_index(drop=True)
combined_df = combined_df.ffill().bfill()
locations = list(combined_df['Location'].unique())

# ML MODELS (K-MEANS and RANDOM FORREST)
print("Training Random Forest and K-Means models... (This might take a few seconds)")

features = ['AT', 'BP', 'SR', 'WS', 'WD', 'NO', 'NO2', 'SO2', 'Ozone', 'CO', 'Benzene', 'NH3', 'NOx']
available_features = [f for f in features if f in combined_df.columns]
all_data_columns = available_features + ['PM2.5']

X = combined_df[available_features]
y = combined_df['PM2.5']

indices = combined_df.index
X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(X, y, indices, test_size=0.2, random_state=42)

rf_model = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)

global_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f'✅ Model Trained! Global RMSE: {global_rmse:.2f}')

test_results = combined_df.loc[idx_test].copy()
test_results['Predicted PM2.5'] = y_pred
test_results['Actual PM2.5'] = y_test
test_results = test_results.sort_values(by='Datetime')

importance_df = pd.DataFrame({'Feature': available_features, 'Importance': rf_model.feature_importances_})
importance_df = importance_df.sort_values(by='Importance', ascending=True)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
kmeans = KMeans(n_clusters=3, random_state=42, n_init='auto')
combined_df['Cluster'] = kmeans.fit_predict(X_scaled).astype(str)


# Static Plots
template = 'plotly_dark'

# Map in dark
map_df = combined_df.groupby('Location')['PM2.5'].mean().reset_index()
map_df['lat'] = map_df['Location'].map(lambda x: delhi_coords[x]['lat'])
map_df['lon'] = map_df['Location'].map(lambda x: delhi_coords[x]['lon'])

map_fig = px.scatter_mapbox(
    map_df, lat="lat", lon="lon", hover_name="Location",
    hover_data={"PM2.5": ':.2f', "lat": False, "lon": False},
    color="PM2.5", size="PM2.5", size_max=35, #bubbles
    color_continuous_scale=px.colors.sequential.Plasma,
    zoom=10.5, center={"lat": 28.61, "lon": 77.20}, # Centered to Delhi
    height=600, title="Average Overall PM2.5 Heatmap (Delhi)",
    template=template
)
# Positron map style
map_fig.update_layout(mapbox_style="carto-positron", margin={"r":0,"t":40,"l":0,"b":0})

corr_matrix = combined_df[all_data_columns].corr()
heatmap_fig = px.imshow(corr_matrix, text_auto='.2f', aspect='auto', title='Correlation Heatmap', color_continuous_scale='RdBu_r', template=template)

feature_importance_fig = px.bar(importance_df, x='Importance', y='Feature', orientation='h', title='Random Forest Feature Importance', template=template)
feature_importance_fig.update_traces(marker_color='#00d2ff')

cluster_fig = px.scatter(combined_df, x='Datetime', y='PM2.5', color='Cluster', opacity=0.6, title='K-Means Clustering over Time', template=template)

# DASH APP INITILIZATION
app = dash.Dash(__name__, external_stylesheets=[dbc.themes.DARKLY])

app.layout = dbc.Container([
    dbc.Row(dbc.Col(html.H2("Delhi AQI Dashboard", className="text-center mt-4 mb-4 fw-bold text-info"))),

    dbc.Card([
        dbc.CardBody([
            dbc.Tabs([
                # Map
                dbc.Tab(label='🗺️ Monitoring Stations', children=[
                    html.Br(),
                    dcc.Graph(figure=map_fig)
                ]),

                # EDA
                dbc.Tab(label='📊 Data Exploration', children=[
                    html.Br(),
                    html.Label("Select Feature to Explore:", className="fw-bold text-light"),
                    dcc.Dropdown(id='eda-dropdown', options=[{'label': f, 'value': f} for f in all_data_columns], value='PM2.5', clearable=False, className="mb-3 text-dark"),
                    dbc.Row([
                        dbc.Col(dcc.Graph(id='eda-box'), width=6),
                        dbc.Col(dcc.Graph(id='eda-hist'), width=6)
                    ])
                ]),

                # Time Series
                dbc.Tab(label='📈 Time Series Analyses', children=[
                    html.Br(),
                    html.Label("Select Location:", className="fw-bold text-light"),
                    dcc.Dropdown(
                        id='location-dropdown',
                        options=[{'label': 'All Stations (Combined)', 'value': 'All'}] + [{'label': loc, 'value': loc} for loc in locations],
                        value='All', clearable=False, className="mb-3 text-dark"
                    ),
                    dcc.Graph(id='dynamic-time-series')
                ]),

                # Features
                dbc.Tab(label='🧩 Features Maps and Importance', children=[
                    html.Br(),
                    dbc.Row([
                        dbc.Col(dcc.Graph(figure=heatmap_fig), width=6),
                        dbc.Col(dcc.Graph(figure=feature_importance_fig), width=6)
                    ])
                ]),

                # Ml outputs
                dbc.Tab(label='🤖 ML Output', children=[
                    html.Br(),
                    dbc.Row([
                        dbc.Col([
                            html.Label("Filter ML Results by Location:", className="fw-bold text-light"),
                            dcc.Dropdown(
                                id='ml-location-dropdown',
                                options=[{'label': 'All Stations (Combined)', 'value': 'All'}] + [{'label': loc, 'value': loc} for loc in locations],
                                value='All', clearable=False, className="mb-3 text-dark"
                            )
                        ], width=6),
                        dbc.Col([
                            html.H5(id='ml-rmse-text', className="text-success text-center fw-bold mt-4")
                        ], width=6)
                    ]),
                    dbc.Row([
                        dbc.Col(dcc.Graph(id='ml-scatter'), width=6),
                        dbc.Col(dcc.Graph(id='ml-line'), width=6)
                    ])
                ]),

                # Clustering Analysis
                dbc.Tab(label='🧬 Clustering Analysis', children=[
                    html.Br(),
                    dcc.Graph(figure=cluster_fig)
                ])
            ], className="nav-fill")
        ])
    ], className="shadow-lg mb-4 bg-dark")
], fluid=True, style={'maxWidth': '1400px'})

# DYNAMIC CALLBACKS
# Callback 1: For EDA (Boxplot & Histogram)
@app.callback(
    [Output('eda-box', 'figure'), Output('eda-hist', 'figure')],
    [Input('eda-dropdown', 'value')]
)
def update_eda(selected_feature):
    box = px.box(combined_df, x='Location', y=selected_feature, color='Location', title=f'{selected_feature} Distributions by Location', template='plotly_dark')
    hist = px.histogram(combined_df, x=selected_feature, nbins=50, title=f'Overall Distribution of {selected_feature}', template='plotly_dark', color_discrete_sequence=['#00d2ff'])
    return box, hist

# Callback 2: For Time Series
@app.callback(
    Output('dynamic-time-series', 'figure'),
    [Input('location-dropdown', 'value')]
)
def update_time_series(selected_location):
    if selected_location == 'All':
        fig = px.line(combined_df, x='Datetime', y='PM2.5', color='Location', title='PM2.5 Trends: All Stations Combined')
    else:
        location_data = combined_df[combined_df['Location'] == selected_location]
        fig = px.line(location_data, x='Datetime', y='PM2.5', title=f'PM2.5 Trends: {selected_location}')
        fig.update_traces(line_color='#00d2ff')

    fig.update_layout(template='plotly_dark', hovermode='x unified')
    return fig

# Callback 3: For ML Outputs
@app.callback(
    [Output('ml-scatter', 'figure'), Output('ml-line', 'figure'), Output('ml-rmse-text', 'children')],
    [Input('ml-location-dropdown', 'value')]
)
def update_ml_tab(selected_location):
    if selected_location == 'All':
        df = test_results
        title_suffix = "(All Stations)"
        color_col = 'Location'
    else:
        df = test_results[test_results['Location'] == selected_location]
        title_suffix = f"({selected_location})"
        color_col = None

    # Specific RMSE calculation
    rmse = np.sqrt(mean_squared_error(df['Actual PM2.5'], df['Predicted PM2.5']))
    rmse_text = f" Current View RMSE: {rmse:.2f}"

    # Scatter Plot
    scatter = px.scatter(df, x='Actual PM2.5', y='Predicted PM2.5', color=color_col if color_col else None, opacity=0.6, title=f'Actual vs Predicted {title_suffix}', template='plotly_dark')
    scatter.add_trace(go.Scatter(x=[df['Actual PM2.5'].min(), df['Actual PM2.5'].max()], y=[df['Actual PM2.5'].min(), df['Actual PM2.5'].max()], mode='lines', name='Perfect Prediction', line=dict(color='#ff003c', dash='dash')))

    # Line Tracking Plot of last 500 Datapoints
    plot_df = df.tail(500)

    if selected_location == 'All':
        # Track by Location
        line = px.line(plot_df, x='Datetime', y='Predicted PM2.5', color='Location', title=f'Predicted PM2.5 Tracking - Last 500 Samples {title_suffix}', template='plotly_dark')
    else:
        # Direct Actual vs Predicted
        line = px.line(plot_df, x='Datetime', y=['Actual PM2.5', 'Predicted PM2.5'], title=f'Actual vs Predicted Tracking - Last 500 Samples {title_suffix}', template='plotly_dark')

    return scatter, line, rmse_text

# RUN APP
if __name__ == '__main__':
    from google.colab import output
    import logging

    log = logging.getLogger('werkzeug')
    log.setLevel(logging.ERROR)

    print("\n" + "="*60)
    print("LINK TO OPEN DASHBOARD:")
    print(output.eval_js("google.colab.kernel.proxyPort(8050)"))
    print("="*60 + "\n")

    app.run(port=8050, debug=False)

Training Random Forest and K-Means models... (This might take a few seconds)
✅ Model Trained! Global RMSE: 36.81

LINK TO OPEN DASHBOARD:
https://8050-gpu-t4-s-kkb-usw4a1-82ranz90z4nt-a.us-west4-1.prod.colab.dev

Dash is running on http://127.0.0.1:8050/



INFO:dash.dash:Dash is running on http://127.0.0.1:8050/



 * Serving Flask app '__main__'
 * Debug mode: off
